In [24]:
import os
import shutil
import kagglehub

# Download dataset from Kaggle
cache_path = kagglehub.dataset_download("nikhilmehrahr26/gurgaon-real-estate-dataset")

# Ensure local data directory exists
os.makedirs('data', exist_ok=True)

# Copy CSV to local data directory
for file in os.listdir(cache_path):
    if file.endswith('.csv'):
        shutil.copy(os.path.join(cache_path, file), os.path.join('data', 'gurgaon_properties.csv'))
        print("Dataset ready in data/gurgaon_properties.csv")
        break


Dataset ready in data/gurgaon_properties.csv


In [25]:
import pandas as pd
import numpy as np

# Load the dataset
df = pd.read_csv('data/gurgaon_properties.csv')

# Inspect dataset dimensions
print("Dataset Shape:", df.shape)



Dataset Shape: (19515, 12)


In [26]:
# View the first 5 records
df.head()





,Price,Status,Area,Rate per sqft,Property Type,Locality,Builder Name,RERA Approval,BHK_Count,Socity,Company Name,Flat Type
0,10700000.0,Under Construction,1138,"9,450",2 BHK Apartment in M3M Antalya Hills Phase I,Sector 79,home,Approved by RERA,2.0,M3M Antalya Hills Phase I,M3M,Apartment
1,14400000.0,Under Construction,1528,"9,450",3 BHK Apartment in M3M Antalya Hills Phase I,Sector 79,Property In Gurgaon,Approved by RERA,3.0,M3M Antalya Hills Phase I,M3M,Apartment
2,10700000.0,Under Construction,1138,"9,450",2 BHK Apartment in M3M Antalya Hills Phase I,Sector 79,properties for sale in Gurgaon,Approved by RERA,2.0,M3M Antalya Hills Phase I,M3M,Apartment
3,40000000.0,Ready to move,4500,"8,888",4 BHK Independent Floor,Sector 57,MM India Pvt Ltd,Not approved by RERA,4.0,Outside Socity,Outside,Plot
4,24000000.0,Under Construction,1800,"13,333",3 BHK Independent Floor in Anant Raj Estate Plots,Sector 63,MM India Pvt Ltd,Approved by RERA,3.0,Anant Raj Estate Plots,Anant,Floor


In [27]:
# Review column names, data types, and non-null counts
print("\n--- Column Summary ---")
df.info()


--- Column Summary ---
<class 'pandas.DataFrame'>
RangeIndex: 19515 entries, 0 to 19514
Data columns (total 12 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Price          19515 non-null  str    
 1   Status         19515 non-null  str    
 2   Area           19515 non-null  int64  
 3   Rate per sqft  19515 non-null  str    
 4   Property Type  19515 non-null  str    
 5   Locality       19515 non-null  str    
 6   Builder Name   19515 non-null  str    
 7   RERA Approval  19515 non-null  str    
 8   BHK_Count      19515 non-null  float64
 9   Socity         19515 non-null  str    
 10  Company Name   19515 non-null  str    
 11  Flat Type      19515 non-null  str    
dtypes: float64(1), int64(1), str(10)
memory usage: 4.5 MB


In [28]:
# Check for missing values across columns
print("\n--- Missing Values Count ---")
print(df.isnull().sum())


--- Missing Values Count ---
Price            0
Status           0
Area             0
Rate per sqft    0
Property Type    0
Locality         0
Builder Name     0
RERA Approval    0
BHK_Count        0
Socity           0
Company Name     0
Flat Type        0
dtype: int64


In [29]:
# Convert Price to numeric and create a Price_Crore column (1 Cr = 10,000,000)
df['Price'] = pd.to_numeric(df['Price'], errors='coerce')
df['Price_Crore'] = (df['Price'] / 1e7).round(2)

# Convert BHK_Count to integer
df['BHK_Count'] = df['BHK_Count'].astype(int)

# Check the converted columns
df[['Price', 'Price_Crore', 'BHK_Count']].head()

,Price,Price_Crore,BHK_Count
0,10700000.0,1.07,2
1,14400000.0,1.44,3
2,10700000.0,1.07,2
3,40000000.0,4.00,4
4,24000000.0,2.40,3


In [30]:
# Strip unnecessary whitespace from Locality
df['Locality'] = df['Locality'].astype(str).str.strip()

# Filter realistic residential boundaries
df_clean = df[
    (df['Area'] >= 300) & (df['Area'] <= 10000) &
    (df['Price_Crore'] >= 0.15) & (df['Price_Crore'] <= 25.0) &
    (df['BHK_Count'] >= 1) & (df['BHK_Count'] <= 6)
].copy()

# Compare row counts before and after filtering
print(f"Original listings count: {len(df)}")
print(f"Cleaned listings count: {len(df_clean)}")

Original listings count: 19515
Cleaned listings count: 17804


In [31]:
# Identify sectors with at least 15 listings
locality_counts = df_clean['Locality'].value_counts()
popular_localities = locality_counts[locality_counts >= 15].index

# Group low-frequency localities into 'Other'
df_clean['Locality'] = df_clean['Locality'].apply(lambda x: x if x in popular_localities else 'Other')

print(f"Total unique localities remaining: {df_clean['Locality'].nunique()}")
print("\nTop 10 Sectors/Localities:")
print(df_clean['Locality'].value_counts().head(10))

Total unique localities remaining: 107

Top 10 Sectors/Localities:
Locality
Sector 65     1178
Sector 79      857
Sector 63      703
Sector 67      638
Sector 106     613
Sector 37D     577
Sector 102     529
Sector 36A     520
Sector 108     518
Sector 113     505
Name: count, dtype: int64


In [32]:
from sklearn.model_selection import train_test_split

# 1. Select independent features (X) and target variable (y)
features = ['Locality', 'BHK_Count', 'Area', 'Flat Type', 'Status']
target = 'Price_Crore'

X = df_clean[features]
y = df_clean[target]

# 2. Split into 80% Training and 20% Testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Features: {features}")
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

Features: ['Locality', 'BHK_Count', 'Area', 'Flat Type', 'Status']
X_train shape: (14243, 5)
X_test shape: (3561, 5)


In [33]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# Define categorical and numerical columns
categorical_cols = ['Locality', 'Flat Type', 'Status']
numerical_cols = ['BHK_Count', 'Area']

# Setup ColumnTransformer with OneHotEncoder
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_cols),
        ('num', 'passthrough', numerical_cols)
    ]
)

print("Preprocessor configured successfully.")

Preprocessor configured successfully.


In [34]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_absolute_error

# Define candidates
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "XGBoost": XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
}

# Train and evaluate
results = {}

for name, model in models.items():
    pipe = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    
    r2 = r2_score(y_test, preds)
    mae = mean_absolute_error(y_test, preds)
    results[name] = {"R2 Score": round(r2, 4), "MAE (Cr)": round(mae, 4), "pipeline": pipe}
    print(f"{name} -> R2 Score: {r2:.4f} | MAE: ₹{mae:.2f} Cr")

Linear Regression -> R2 Score: 0.7993 | MAE: ₹0.92 Cr
Random Forest -> R2 Score: 0.8889 | MAE: ₹0.48 Cr
XGBoost -> R2 Score: 0.8579 | MAE: ₹0.73 Cr


In [35]:
import joblib

# Select the best performing model pipeline (Random Forest)
best_pipeline = results["Random Forest"]["pipeline"]

# Package pipeline and categorical values for the web application
model_artifact = {
    "pipeline": best_pipeline,
    "localities": sorted(df_clean['Locality'].unique().tolist()),
    "flat_types": sorted(df_clean['Flat Type'].unique().tolist()),
    "statuses": sorted(df_clean['Status'].unique().tolist())
}

# Save compressed model artifact
joblib.dump(model_artifact, "model.pkl", compress=3)
print("Model pipeline and metadata successfully saved to model.pkl")

# Sample prediction test
sample_property = pd.DataFrame([{
    'Locality': 'Sector 65',
    'BHK_Count': 3,
    'Area': 1800,
    'Flat Type': 'Apartment',
    'Status': 'Ready to move'
}])

sample_val = best_pipeline.predict(sample_property)[0]
print(f"Sample prediction (Sector 65, 3 BHK, 1800 sq.ft.): ₹{sample_val:.2f} Cr")


Model pipeline and metadata successfully saved to model.pkl
Sample prediction (Sector 65, 3 BHK, 1800 sq.ft.): ₹2.55 Cr
